# Example run

A live Sera swarm: GPU check → credentials → baseline → investigation → measured comparison → usable runner.

This notebook starts with **no configured API keys and no saved outputs**. Run one cell at a time. Secret prompts accept keys at runtime; never paste a key into code. Do not use **Run All** for the first rehearsal.

Use a dedicated notebook kernel. The helpers temporarily set process credentials during calls; this is not a shared multi-user server.


## 0. Prepare the notebook

Open this notebook from a checkout of the Sera repository. The `experiments/` and `benchmarks/` directories must be present. Install that checkout with the `swarm` and `litellm` extras if needed (`python -m pip install -e '.[swarm,litellm]'`). Do not upgrade the tested GPU stack during the demo; see `docs/gpu-runtime-contract.md`.

The measured target is **Qwen2.5-72B**, revision pinned by Sera. Download its weights before recording. The saved download and validation took 4m53s; cached startup took 60–71s. A fresh Molab runtime can lose cached files. Download time is not inference latency.


In [ ]:
import sys
from pathlib import Path
repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiments/example_run.py').is_file()), None)
if repo is None:
    raise RuntimeError('Open this notebook inside the Sera repository checkout')
sys.path.insert(0, str(repo))
from experiments.example_run import ExampleRun
from getpass import getpass
example = ExampleRun()
example.status()

## 1. GPU check — no keys needed

Without a GPU, stop here and **Change runtime to GPU**. A detected GPU is not proof that its runtime supports FP8. Sera validates the runtime before running. Other active GPU jobs must finish first.


In [ ]:
gpu = example.gpu()
print(gpu)
if not gpu['available']:
    raise RuntimeError('Change runtime to GPU')

## 2. Confirm the blank state

This intentional no-key check must stop before network or GPU work. Even if the kernel has an old environment key, this session does not use it until you explicitly configure credentials.


In [ ]:
try:
    example.check_weave()
except ValueError as error:
    print('Expected blank-state stop:', error)
else:
    raise RuntimeError('Expected no configured credentials; start a fresh kernel')

## 3. Add only the W&B key, then test Weave

Use an authorized `entity/project`. The key is entered through a hidden prompt, not source code. This test writes one small Weave connection trace, uses no GPU, and does not call an investigator model.


In [ ]:
example.configure_weave(getpass('W&B API key (hidden): '), project=input('Weave entity/project: ').strip())
example.status()

In [ ]:
example.check_weave()

## 4. Select and test the investigator provider

Choose one route:

- `wandb`: reuses the W&B key; enter an available model ID, such as `openai/gpt-oss-120b`. Inference credits are separate from GPU access.
- `litellm-openai`: calls OpenAI through the LiteLLM SDK. Enter `gpt-6-astra` and an OpenAI API key. This does not need the Codex controller. Model access depends on the key's project.
- `codex-relay`: for the Astra demonstration, enter `gpt-6-astra` and the configured relay directory. A matching logged-in external controller must already be running. This route is not self-contained hosting.
- `openai-compatible`: enter an operator-approved HTTPS API or LiteLLM gateway, its model ID, and its key. This uses a second key; the W&B key remains for Weave. Do not use untrusted endpoints.

W&B's published model list does not include Astra. Naming a model does not grant access. A compatible endpoint must support the strict output contract; Sera does not silently weaken it.


In [ ]:
provider = input('Provider: wandb / litellm-openai / codex-relay / openai-compatible: ').strip()
agent_model = input('Investigator model ID: ').strip()
provider_options = {}
if provider == 'openai-compatible':
    provider_options = {'base_url': input('Approved HTTPS base URL: ').strip(), 'api_key': getpass('Provider key (hidden): ')}
elif provider == 'litellm-openai':
    provider_options = {'api_key': getpass('OpenAI API key (hidden): ')}
elif provider == 'codex-relay':
    provider_options = {'relay_dir': input('Controller relay directory: ').strip()}
try:
    example.configure_agent(provider=provider, model=agent_model, **provider_options)
finally:
    provider_options.clear()
example.status()

This is the first investigator-model test. It makes **34 format/evidence checks**, with at most one retry per check, and uses **no GPU trials**. A failure stops the workflow and preserves the failure record. A pass establishes format compatibility, not reasoning quality. Changing endpoint, model, or schema requires another matching check.


In [ ]:
example.check_agent()

## 5. Measure the baseline

Qwen72B BF16 weights exceed this card's memory; that is an estimate, not an executed BF16 baseline. Our measured baseline uses **FP8 weights + BF16 KV**, 4,096 context/batch tokens, eight maximum sequences, and the pinned defaults.

The eight fixed tasks are measured at concurrency 1, 2, 4, and 8. Quality floor: 99%. The size metric below is **sampled peak total GPU memory**, not weight-file size. This preview baseline closes its runner before optimization to free the GPU.


In [ ]:
baseline_metrics = example.baseline()
baseline_metrics

## 6. Run the autonomous swarm

Three investigators inspect evidence, share findings, propose changes, and send them to the arbiter. Sera validates each experiment, measures it, and feeds the outcome into the next round. It stops on the progress-plus-confirmation rule or when no legal experiment remains—not a fixed total trial cap.

Latency must improve by at least 5% while the fixed quality gate passes. This call **remeasures its own baseline**; the final percentage uses that same-run baseline, not the preview above. No improvement is a valid result.

The recorded Astra investigation took about **14 minutes for three rounds**, excluding initial deployment and final runner restoration. This is a historical timing, not a promise for this run. The call keeps the notebook busy; follow its Weave link and saved run records.


In [ ]:
optimized = example.optimize()
optimized.print_summary()
print('Weave:', optimized.weave_url)

## 7. Compare measured results

p95 is the worst per-load p95, not a pooled percentile. Startup is separate. A missing or rejected result is not shown as a speedup. These easy repeated tasks are a controlled demo, not proof of general model quality or search superiority.


In [ ]:
comparison = example.compare(optimized)
comparison

## 8. Use the returned runner, then release the GPU

This extra request checks the first unchanged task. It does not establish unseen-task quality. If no configuration passed, there is no runner to use. Always execute cleanup, including after stopping a recording.


In [ ]:
if not optimized.models:
    print('No safe configuration returned; inspect the report.')
else:
    prompts, evaluate = example._tasks()
    response = optimized.models[0].generate(prompts[0])
    print({'output': response.text, 'task_passed': evaluate(prompts[0], response.text), 'latency_ms': response.latency_ms})

In [ ]:
example.close()

## 9. Clear credentials before sharing

Restart the kernel to remove remaining runtime objects and cached SDK state. Clear all notebook outputs before saving or sharing. Do not commit any executed copy containing user task data. The repository copy is intentionally unexecuted.

For a recorded fallback, show the saved Astra trace as **previously measured**, never as current live output. The historical 19.55% gain belongs to its specific workload. Joint-workload latency acceptance and generic production readiness are separate claims.


In [ ]:
example.clear_credentials()